# Gold · Dimensión de Clientes

## Propósito

Construir la dimensión de clientes en la capa Gold con historial de cambios (SCD2), de modo que cada pedido pueda analizarse con los atributos que el cliente tenía en ese momento.

## Flujo

1. Leer la versión actual de clientes desde la capa Silver.
2. Descartar el campo de contraseña: es un dato sensible que no debe llegar a Gold.
3. En la primera carga, crear la tabla con todos los clientes como versión vigente.
4. En las siguientes ejecuciones, comparar cada cliente contra su versión vigente.
5. Cerrar las versiones anteriores de los clientes que cambiaron.
6. Insertar las nuevas versiones de los clientes nuevos y de los que cambiaron.

## Destino

Capa Gold.

**Tabla:** `gld_dataco.supply_chain.desc_dim_customers`

**Llave sustituta:** `cliente_key`

## Atributos que generan una nueva versión

* Nombre y apellido
* Email
* Segmento
* Ciudad, estado, calle, código postal y país

## Control de vigencia

* `es_vigente` indica la versión actual de cada cliente.
* Las consultas de negocio filtran `es_vigente = true`.

## Resultado esperado

La dimensión de clientes actualizada, con el historial de cambios conservado y una única versión vigente por cliente.

In [0]:
# ============================================================
# DIM_CLIENTE — Celda 1: Parámetros
# ============================================================

dbutils.widgets.text("silver_catalog", "slv_dataco", "Catálogo Silver")
dbutils.widgets.text("silver_schema", "supply_chain", "Esquema Silver")

dbutils.widgets.text("gold_catalog", "gld_dataco", "Catálogo Gold")
dbutils.widgets.text("gold_schema", "supply_chain", "Esquema Gold")

silver_catalog = dbutils.widgets.get("silver_catalog")
silver_schema = dbutils.widgets.get("silver_schema")
gold_catalog = dbutils.widgets.get("gold_catalog")
gold_schema = dbutils.widgets.get("gold_schema")

SILVER_CLIENTES_FQN = f"{silver_catalog}.{silver_schema}.md_clientes"
GOLD_DIM_CLIENTE_FQN = f"{gold_catalog}.{gold_schema}.desc_dim_customers"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_catalog}.{gold_schema}")

print(f"✓ Origen: {SILVER_CLIENTES_FQN}")
print(f"✓ Destino: {GOLD_DIM_CLIENTE_FQN}")

In [0]:
# ============================================================
# DIM_CLIENTE — Celda 2: INICIO DE LOG DE EJECUCIÓN — gld_dim_cliente
# ============================================================

import uuid

id_ejecucion = dbutils.jobs.taskValues.get(
    taskKey="prepare_pipeline_control",
    key="id_ejecucion",
    debugValue=""
)

LOG_ID = str(uuid.uuid4())
NOTEBOOK = "gld_dim_customers"
CAPA = "gold"

spark.sql(f"""
INSERT INTO metadata_dataco.supply_chain.hd_etl_log
(log_id, id_ejecucion, notebook, capa, estado, fecha_inicio)
VALUES ('{LOG_ID}', '{id_ejecucion}', '{NOTEBOOK}', '{CAPA}', 'EN_PROCESO', current_timestamp())
""")

dbutils.jobs.taskValues.set(key="log_id", value=LOG_ID)

print(f"✓ Registro EN_PROCESO creado en etl_log: {LOG_ID}")

In [0]:
# ============================================================
# DIM_CLIENTE — Celda 3: Leer versión actual de Silver
# ============================================================

from pyspark.sql import functions as F

df_clientes_actual = spark.table(SILVER_CLIENTES_FQN).select(
    "customer_id",
    "customer_fname",
    "customer_lname",
    "customer_email",
    "customer_segment",
    "customer_city",
    "customer_state",
    "customer_street",
    "customer_zipcode",
    "customer_country"
)

# Nunca debe llegar a Gold — es un dato sensible que no debería
# existir en Silver siquiera. Se descarta explícitamente aquí.
# (customer_password NO se selecciona arriba a propósito)

print(f"✓ Clientes en Silver: {df_clientes_actual.count()}")

In [0]:
# ============================================================
# DIM_CLIENTE — Celda 4: Crear tabla si no existe (primera carga)
# ============================================================

from delta.tables import DeltaTable

if not spark.catalog.tableExists(GOLD_DIM_CLIENTE_FQN):

    df_inicial = (
        df_clientes_actual
        .withColumn("cliente_key", F.monotonically_increasing_id())
        .withColumn("fecha_inicio_vigencia", F.current_date())
        .withColumn("fecha_fin_vigencia", F.lit(None).cast("date"))
        .withColumn("es_vigente", F.lit(True))
    )

    (
        df_inicial.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(GOLD_DIM_CLIENTE_FQN)
    )

    print(f"✓ Tabla {GOLD_DIM_CLIENTE_FQN} creada con {df_inicial.count()} clientes (carga inicial)")
    CARGA_INICIAL = True
else:
    print(f"✓ Tabla {GOLD_DIM_CLIENTE_FQN} ya existe — se procesa como SCD2")
    CARGA_INICIAL = False

In [0]:
# ============================================================
# DIM_CLIENTE — Celda 5: Lógica SCD2 (solo si la tabla ya existía)
# ============================================================

if not CARGA_INICIAL:

    delta_dim_cliente = DeltaTable.forName(spark, GOLD_DIM_CLIENTE_FQN)

    df_vigente = (
        delta_dim_cliente.toDF()
        .filter(F.col("es_vigente") == True)
    )

    # Columnas que, si cambian, generan una nueva versión del cliente
    COLUMNAS_COMPARADAS = [
        "customer_fname", "customer_lname", "customer_email",
        "customer_segment", "customer_city", "customer_state",
        "customer_street", "customer_zipcode", "customer_country"
    ]

    condicion_cambio = " OR ".join([
        f"v.{c} <> n.{c}" for c in COLUMNAS_COMPARADAS
    ])

    df_comparado = (
        df_clientes_actual.alias("n")
        .join(df_vigente.alias("v"), on="customer_id", how="left")
        .selectExpr(
            "n.*",
            "v.cliente_key AS cliente_key_existente",
            f"CASE WHEN v.customer_id IS NULL THEN 'NUEVO' "
            f"WHEN {condicion_cambio} THEN 'CAMBIO' "
            f"ELSE 'SIN_CAMBIO' END AS tipo_evento"
        )
    )

    df_a_cerrar = df_comparado.filter(F.col("tipo_evento") == "CAMBIO")
    df_a_insertar = df_comparado.filter(F.col("tipo_evento").isin("NUEVO", "CAMBIO"))

    print(f"✓ Clientes nuevos: {df_comparado.filter(F.col('tipo_evento') == 'NUEVO').count()}")
    print(f"✓ Clientes con cambios (nueva versión): {df_a_cerrar.count()}")
    print(f"✓ Clientes sin cambios: {df_comparado.filter(F.col('tipo_evento') == 'SIN_CAMBIO').count()}")

In [0]:
# ============================================================
# DIM_CLIENTE — Celda 6: Cerrar versiones viejas (solo los que cambiaron)
# ============================================================

if not CARGA_INICIAL and df_a_cerrar.count() > 0:

    ids_con_cambio = [row["customer_id"] for row in df_a_cerrar.select("customer_id").collect()]

    delta_dim_cliente.update(
        condition=(F.col("customer_id").isin(ids_con_cambio)) & (F.col("es_vigente") == True),
        set={
            "es_vigente": F.lit(False),
            "fecha_fin_vigencia": F.current_date()
        }
    )

    print(f"✓ {len(ids_con_cambio)} versiones anteriores cerradas")
elif not CARGA_INICIAL:
    print("✓ No hay versiones que cerrar")

In [0]:
# ============================================================
# DIM_CLIENTE — Celda 7: Insertar nuevas versiones (nuevos + cambiados)
# ============================================================

if not CARGA_INICIAL and df_a_insertar.count() > 0:

    max_key = spark.sql(f"SELECT MAX(cliente_key) AS m FROM {GOLD_DIM_CLIENTE_FQN}").collect()[0]["m"] or 0

    df_nuevas_versiones = (
        df_a_insertar
        .drop("cliente_key_existente", "tipo_evento")
        .withColumn("cliente_key", max_key + F.monotonically_increasing_id() + 1)
        .withColumn("fecha_inicio_vigencia", F.current_date())
        .withColumn("fecha_fin_vigencia", F.lit(None).cast("date"))
        .withColumn("es_vigente", F.lit(True))
    )

    df_nuevas_versiones.write.format("delta").mode("append").saveAsTable(GOLD_DIM_CLIENTE_FQN)

    print(f"✓ {df_nuevas_versiones.count()} nuevas versiones insertadas")
elif not CARGA_INICIAL:
    print("✓ No hay nuevas versiones que insertar")

In [0]:
TABLA_DESTINO = "gld_dataco.supply_chain.desc_dim_customers"   # cambia por notebook (ver tabla)
filas_procesadas = spark.table(TABLA_DESTINO).count()

spark.sql(f"""
UPDATE metadata_dataco.supply_chain.hd_etl_log
SET estado = 'EXITOSO',
    fecha_fin = current_timestamp(),
    filas_procesadas = {filas_procesadas}
WHERE log_id = '{LOG_ID}'
""")

print(f"✓ etl_log actualizado: EXITOSO ({filas_procesadas} filas en {TABLA_DESTINO})")